In [1]:
# using linear regression on a real dataset

# California Housing Prices
# https://www.kaggle.com/datasets/camnugent/california-housing-prices

In [2]:
# import library
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("./dataset/housing.csv")

In [4]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [5]:
# data cleaning: missing value
df.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [6]:
#Replacing missing values for numeric attributes using mean


df.fillna({'total_bedrooms': df['total_bedrooms'].mean()}, inplace=True)

In [7]:
# look at the categorical feature
df["ocean_proximity"].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

In [8]:
# convert these categories from text to numbers

# method 1 : OrdinalEncoder
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html#sklearn.preprocessing.OrdinalEncoder

from sklearn.preprocessing import OrdinalEncoder

df_cat = df[["ocean_proximity"]]
ordinal_encoder = OrdinalEncoder()
df_cat_encoded = ordinal_encoder.fit_transform(df_cat)

In [9]:
df_cat_encoded

array([[3.],
       [3.],
       [3.],
       ...,
       [1.],
       [1.],
       [1.]])

In [10]:
# get the list of categories using the categories_ instance variable.
ordinal_encoder.categories_

[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
       dtype=object)]

In [11]:
# convert these categories from text to numbers

# method 2 : one-hot encoding
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html#sklearn.preprocessing.OneHotEncoder

from sklearn.preprocessing import OneHotEncoder
cat_encoder = OneHotEncoder()
df_cat_1hot = cat_encoder.fit_transform(df_cat)

In [12]:
df_cat_1hot

# the output is a SciPy sparse matrix
# the matrix is full of zeros except for a single 1 per row


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 20640 stored elements and shape (20640, 5)>

In [13]:
# toarray() : convert it to a (dense) NumPy array
df_cat_1hot.toarray()

array([[0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0.],
       ...,
       [0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0.]])

In [14]:
df_cat_1hot.toarray().shape

(20640, 5)

In [15]:
ordinal_encoder.categories_

[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
       dtype=object)]

In [16]:
# array to df
df_cat_attr = pd.DataFrame(df_cat_1hot.toarray(), columns=['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY','NEAR OCEAN'])

In [17]:
df_cat_attr

,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...
20635,0.0,1.0,0.0,0.0,0.0
20636,0.0,1.0,0.0,0.0,0.0
20637,0.0,1.0,0.0,0.0,0.0
20638,0.0,1.0,0.0,0.0,0.0


In [18]:
# Re-organizing the dataset
# The target value (median_house_value) should place in the last column
# df.iloc[]: Purely integer-location based indexing for selection by position (from 0 to length-1 of the axis).

df_num_attr = df.iloc[:,:8] # numerical features
df_tar = df.iloc[:,-2:-1] # target variable

In [19]:
df_num_attr.shape, df_cat_attr.shape, df_tar.shape

((20640, 8), (20640, 5), (20640, 1))

In [20]:
df_prepare = pd.concat([df_num_attr, df_cat_attr,df_tar], axis =1)

In [21]:
df_prepare

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN,median_house_value
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,0.0,0.0,0.0,1.0,0.0,452600.0
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,0.0,0.0,0.0,1.0,0.0,358500.0
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,0.0,0.0,0.0,1.0,0.0,352100.0
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,0.0,0.0,0.0,1.0,0.0,341300.0
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,0.0,0.0,0.0,1.0,0.0,342200.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,0.0,1.0,0.0,0.0,0.0,78100.0
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,0.0,1.0,0.0,0.0,0.0,77100.0
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,0.0,1.0,0.0,0.0,0.0,92300.0
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,0.0,1.0,0.0,0.0,0.0,84700.0


In [22]:
df_prepare.shape

(20640, 14)

In [23]:
# Selecting features
# divide the given columns into two types of variables 
# dependent(or target variable) and independent variable(or feature variables)


X = df_prepare.iloc[:,:-1] # Features
y = df_prepare.iloc[:,-1:] # Target variable

In [24]:
# Splitting the dataset into a training set and a test set
# train_test_split(), pass 3 parameters: features, target, and test_set size
# random_state to select records randomly

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=16)

In [25]:
X_train.shape,X_test.shape, y_train.shape, y_test.shape

((15480, 13), (5160, 13), (15480, 1), (5160, 1))

In [26]:
# train the linear model with holdout method: 
# splitting the dataset into two parts: one for training and one for testingjust split dataset to 

from sklearn import linear_model

# instantiate the model (using the default parameters)
reg = linear_model.LinearRegression()

# fit the model with training data
reg.fit(X_train, y_train)

# get model predictions for test data
y_pred = reg.predict(X_test)

In [27]:
# Model Evaluation 
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred) 
mse = mean_squared_error(y_test, y_pred) 
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

In [28]:
mae, mse, rmse, r2 

(49790.53674724027, 4847309034.5069275, 69622.61869900419, 0.6376142217103595)

In [29]:
#  Cross validation 
#  https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html

from sklearn.model_selection import cross_validate

scores = cross_validate(reg, X, y, cv=10,
                        scoring=('neg_mean_absolute_error','neg_mean_squared_error','neg_root_mean_squared_error','r2'),
                        return_train_score=True)


In [30]:
scores.keys()

dict_keys(['fit_time', 'score_time', 'test_neg_mean_absolute_error', 'train_neg_mean_absolute_error', 'test_neg_mean_squared_error', 'train_neg_mean_squared_error', 'test_neg_root_mean_squared_error', 'train_neg_root_mean_squared_error', 'test_r2', 'train_r2'])

In [31]:
test_scores = scores['test_r2']
train_scores = scores['test_r2']

# calculate the average scores for 10 fold 
print('\n10-fold CV r2_scores:')
print(f'training score = {np.mean(train_scores)} +/- {np.std(train_scores)}')
print(f'testing score = {np.mean(test_scores)} +/- {np.std(test_scores)}')


10-fold CV r2_scores:
training score = 0.514103398268315 +/- 0.12413528382434344
testing score = 0.514103398268315 +/- 0.12413528382434344
